# 실험3-K — carry(전달) vs no-carry, 둘 다 +TE, **K 스윕** (해준 담당)

은지님 K sweep(bimamba+carry+TE)에 맞춰 K별로 **학습**하고, 각 K에서 **carry 전달 on/off**(둘 다 +TE, overlap OFF)를
eval → 교수님 질문 "carry 전달이 TE 위에서 값어치 있나"를 K별로 확인.

- **학습 대상 = bimamba+carry (chunk_size=K)**. K=100 은 기존 `bimamba` 폴더 재사용(학습 스킵).
- **eval = 같은 ckpt 에 sscp on/off + TE 만 다르게(재학습 X)**. overlap OFF, lr 1e-5.
- ★서버 4대면 맨 위 `K_LIST` 를 자기 몫으로 나눠 각 서버에서 실행. 학습→eval→결과 순.


## 0) 부팅 + K별 태그 등록


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

# ── 무엇을 하나 ────────────────────────────────────────────────────────────
# 은지님 K sweep(bimamba+carry+TE)에 맞춰 K별로 **학습**하고, 각 K에서
#   carry 전달 on/off (둘 다 +TE, overlap OFF)를 eval → carry가 TE 위에서 값어치 있는지.
# 학습 대상 = bimamba+carry (chunk_size=K). eval 은 같은 ckpt 에 sscp on/off + TE 만 다르게(재학습 X).

TASK   = 'libero_10'
K_LIST = [10, 15, 20, 50, 100, 150]   # 은지님과 동일. ★서버 4대면 각자 자기 몫으로 편집:
#   예) 서버1 [10,150] · 서버2 [15] · 서버3 [20,100] · 서버4 [50]   (100 은 기존 bimamba 재사용=학습 스킵)
SEEDS  = [0]                          # 1 seed 로 경향 먼저. 필요시 [0,1].
N_EP   = 100                          # eval task당 (overall 1000). 빠르게 50 도 가능.
GPUS   = v23.available_gpus()

# K=100 은 기존 'bimamba' 폴더(carry, K=100) 재사용 → 학습 스킵. 나머지 K 는 새 태그 등록.
def src_tag(K):
    return 'bimamba' if K == 100 else f'bimamba_k{K}'
for K in K_LIST:
    t = src_tag(K)
    v23.MODEL_CONFIGS.setdefault(t, ('acm2_sscp_literal_bimamba', v23.LR, K,
                                     ['--policy.sscp_enabled=true'], True))  # carry ON, overlap 없음
    v23.MODEL_DIR_NAMES.setdefault(t, t)

_TE = ['--policy.temporal_ensemble_coeff=0.01', '--policy.n_action_steps=1']
# eval 변형: (out 태그 접미사, sscp 플래그)
VARIANTS = [('carry_te',   '--policy.sscp_enabled=true'),   # carry 전달 + TE (우리)
            ('nocarry_te', '--policy.sscp_enabled=false')]  # carry 끔 + TE (baseline)
def out_tag(K, suf):
    return f'bk{K}_{suf}'
for K in K_LIST:
    for suf, _ in VARIANTS:
        v23.MODEL_DIR_NAMES.setdefault(out_tag(K, suf), out_tag(K, suf))

print('K sweep:', K_LIST, '| seeds', SEEDS, '| N_EP', N_EP, '| GPU', GPUS)
print('학습 필요(K!=100):', [src_tag(K) for K in K_LIST if K != 100])

## 1) 학습 (K별 bimamba+carry — 이미 된 것/K=100 은 스킵)


In [ ]:
# ── K별 bimamba+carry 학습 (K=100 은 기존 bimamba 재사용→스킵, 이미 된 것도 스킵/이어서) ──
#   ★서버 4대면 K_LIST 를 나눠서 각 서버가 자기 몫만. 여기선 K_LIST 전체 학습.
train_jobs = [(src_tag(K), s, TASK) for K in K_LIST if K != 100 for s in SEEDS]
if train_jobs:
    cf.run_training_jobs(train_jobs, GPUS, prefetch_task=TASK)
else:
    print('학습할 것 없음 (전부 K=100 재사용 or 이미 완료)')

## 2) eval (carry+TE / nocarry+TE)


In [ ]:
# ── 각 K 체크포인트에서 carry+TE / nocarry+TE eval (재학습 X, sscp on/off + TE). action 기록→떨림 ──
_MINEP = 10 * N_EP // 2
def _has_valid(ot, s):
    info = v23.eval_clean_dir(ot, s, TASK) / 'eval_info.json'
    if not info.exists(): return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

jobs = []
for K in K_LIST:
    for s in SEEDS:
        for suf, flag in VARIANTS:
            ot = out_tag(K, suf)
            if not _has_valid(ot, s):
                jobs.append((K, s, suf, flag, ot))
print(f'eval 실행 {len(jobs)} / 전체 {len(K_LIST)*len(SEEDS)*len(VARIANTS)} (유효 완료 skip)')
ng = len(GPUS)
for i in range(0, len(jobs), ng):
    chunk = jobs[i:i + ng]
    labeled = []
    for g, (K, s, suf, flag, ot) in zip(GPUS, chunk):
        try:
            cmd = v23.make_eval_cmd(src_tag(K), seed=s, task=TASK, gpu_id=g, n_episodes=N_EP,
                                    select=cf.CKPT_STEP, extra_policy=[flag] + _TE,
                                    out_dir=v23.eval_clean_dir(ot, s, TASK))
            labeled.append((f'{ot}/seed{s}', cmd))
        except FileNotFoundError as e:
            print('  skip:', e)
    if labeled:
        print(f'\n===== eval 청크 {i//ng+1} ({len(labeled)} run) =====')
        v23.launch_cmds_live(labeled)
print('\n완료')

## 3) 결과 — K별 SR·떨림


In [ ]:
# ── 결과: K별 carry+TE vs nocarry+TE (SR + 떨림). carry 가 TE 위에서 값어치 있나? ──
import numpy as np, smooth_metrics_paper as smp
importlib.reload(smp)
FS, STRIDE = cf.fps_of(TASK), 100
def rec(tag, s):
    d = cf.OUTPUT_BASE / 'eval_clean' / TASK / v23.MODEL_DIR_NAMES.get(tag, tag) / f'seed{s}'
    if not d.is_dir(): return None
    best = None
    for info in d.rglob('eval_info.json'):
        try: ov = json.loads(info.read_text()).get('overall', {})
        except Exception: continue
        n = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or n > best['n']:
            best = {'sr': ov.get('pc_success'), 'n': n, 'act': (info.parent/'actions').is_dir(), 'p': info.parent}
    return best
def sr_of(ot):
    vals = [rec(ot, s)['sr'] for s in SEEDS if rec(ot, s) and (rec(ot, s)['n'] or 0) >= 10*N_EP//2 and rec(ot, s)['sr'] is not None]
    return float(np.mean(vals)) if vals else None
def sm_of(ot):
    trajs = []
    for s in SEEDS:
        e = rec(ot, s)
        if e and (e['n'] or 0) >= 10*N_EP//2 and e['act']:
            t = v23._load_action_trajs(e['p']/'actions') or []
            if len(t) >= 80: trajs += t
    return smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS) if trajs else None

print('== SR vs K (carry+TE  vs  nocarry+TE) ==')
print(f'{"K":>5}{"carry+TE":>12}{"nocarry+TE":>12}{"Δ(carry-no)":>13}')
for K in K_LIST:
    a, b = sr_of(out_tag(K,'carry_te')), sr_of(out_tag(K,'nocarry_te'))
    da = f'{a-b:+.1f}' if (a is not None and b is not None) else ''
    print(f'{K:>5}{(f"{a:.1f}" if a is not None else "-"):>12}{(f"{b:.1f}" if b is not None else "-"):>12}{da:>13}')

print('\n== 떨림 vs K (jerk / B/I / sflip) ==')
print(f'{"K":>5}  {"variant":<12}{"jerk":>9}{"bnd":>9}{"int":>9}{"B/I":>7}{"sflip":>9}{"n":>6}')
for K in K_LIST:
    for suf, _ in VARIANTS:
        a = sm_of(out_tag(K, suf))
        if a:
            print(f'{K:>5}  {suf:<12}{a["jerk_rms_mean"]:>9.4f}{a["boundary_jerk_rms_mean"]:>9.4f}'
                  f'{a["interior_jerk_rms_mean"]:>9.4f}{a["boundary_interior_ratio_mean"]:>7.2f}'
                  f'{a["sign_flip_rate_mean"]:>9.4f}{a["n_traj"]:>6}')
        else:
            print(f'{K:>5}  {suf:<12}' + ' '*40 + '(빈칸)')
print('\n판단: 어떤 K 에서든 carry+TE 가 nocarry+TE 보다 SR·떨림 확실히 우위면 carry 생존. 비슷하면 carry 제거 검토.')